> 在线讲义：[第 5 章 · Bootstrap 与 Beta 后验："B 比 A 好的概率"怎么算](https://wikieden.github.io/math-and-eval-stats/lessons/05-bootstrap-bayes.html) · 本书《数学与实验统计》

# 第 5 章 · Bootstrap 与 Beta 后验："B 比 A 好的概率"怎么算

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/05-bootstrap-bayes.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 bootstrap | 百分位 | `bootstrap_ci(40, 50)` | [0.680, 0.900] |
| 第二步 后验 | Beta(1+k, 1+n−k) | `beta_posterior(40, 50)`、`credible_interval(40, 50)` | 均值 0.788，[0.669, 0.887] |
| 第三步 P(B > A) | 蒙特卡洛 | `prob_b_better(40, 50, 30, 50)` | 0.984 |
| 第三步 差值区间 | bootstrap | `diff_bootstrap(40, 50, 30, 50)` | [+0.020, +0.380] |
| 第四步 分层 | 任务内重抽 | `stratified_bootstrap_mean(SCORES)` | 0.633，[0.611, 0.655] |

例子：第 2 章的 40/50 vs 30/50；分层 bootstrap 用编的"3 个任务 × 5 个种子"。
里程碑 ME-M4 的 13/20 vs 14/20 的 P(B > A) 留给自己算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.stats import beta
SCORES = np.array([[0.62, 0.70, 0.58, 0.66, 0.64],   # 任务 1，5 个种子（编的）
                   [0.31, 0.45, 0.38, 0.29, 0.40],   # 任务 2
                   [0.88, 0.91, 0.85, 0.93, 0.90]])  # 任务 3

**函数 `bootstrap_ci`**：百分位 bootstrap：把 n 次结果（k 个 1、n − k 个 0）有放回重抽 reps 次，取成功率的分位数。

In [ ]:
def bootstrap_ci(k: int, n: int, reps: int = 20000, seed: int = 0, level: float = 0.95) -> tuple[float, float]:
    """百分位 bootstrap：把 n 次结果（k 个 1、n − k 个 0）有放回重抽 reps 次，取成功率的分位数。"""
    rng = np.random.default_rng(seed)
    data = np.r_[np.ones(k), np.zeros(n - k)]
    means = rng.choice(data, (reps, n), replace=True).mean(1)
    a = (1 - level) / 2
    return float(np.quantile(means, a)), float(np.quantile(means, 1 - a))

**函数 `beta_posterior`**：均匀先验 Beta(1, 1) + k 次成功、n − k 次失败 → 后验 Beta(1 + k, 1 + n − k)。

In [ ]:
def beta_posterior(k: int, n: int, a0: float = 1, b0: float = 1):
    """均匀先验 Beta(1, 1) + k 次成功、n − k 次失败 → 后验 Beta(1 + k, 1 + n − k)。"""
    return beta(a0 + k, b0 + n - k)

**函数 `credible_interval`**：后验的中间 95%（等尾可信区间）。

In [ ]:
def credible_interval(k: int, n: int, level: float = 0.95) -> tuple[float, float]:
    """后验的中间 95%（等尾可信区间）。"""
    d = beta_posterior(k, n)
    a = (1 - level) / 2
    return float(d.ppf(a)), float(d.ppf(1 - a))

**函数 `prob_b_better`**：P(p_B > p_A)：两个后验各抽一个数比大小，数 B 赢的比例。

In [ ]:
def prob_b_better(kb: int, nb: int, ka: int, na: int, draws: int = 400000, seed: int = 0) -> float:
    """P(p_B > p_A)：两个后验各抽一个数比大小，数 B 赢的比例。"""
    rng = np.random.default_rng(seed)
    pb = rng.beta(1 + kb, 1 + nb - kb, draws)
    pa = rng.beta(1 + ka, 1 + na - ka, draws)
    return float(np.mean(pb > pa))

**函数 `diff_bootstrap`**：差值的 bootstrap 区间：两组各自有放回重抽，算 p_B − p_A 的 2.5% 和 97.5% 分位。

In [ ]:
def diff_bootstrap(kb: int, nb: int, ka: int, na: int, reps: int = 20000, seed: int = 0) -> tuple[float, float]:
    """差值的 bootstrap 区间：两组各自有放回重抽，算 p_B − p_A 的 2.5% 和 97.5% 分位。"""
    rng = np.random.default_rng(seed)
    b = rng.binomial(nb, kb / nb, reps) / nb
    a = rng.binomial(na, ka / na, reps) / na
    d = b - a
    return float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))

**函数 `stratified_bootstrap_mean`**：Agarwal 2021 的分层 bootstrap：每个任务内部对种子有放回重抽，再对任务取平均。返回 (点估计, 下界, 上界)。

In [ ]:
def stratified_bootstrap_mean(scores: np.ndarray, reps: int = 20000, seed: int = 0) -> tuple[float, float, float]:
    """Agarwal 2021 的分层 bootstrap：每个任务内部对种子有放回重抽，再对任务取平均。返回 (点估计, 下界, 上界)。"""
    rng = np.random.default_rng(seed)
    t, s = scores.shape
    idx = rng.integers(0, s, (reps, t, s))
    res = np.take_along_axis(np.broadcast_to(scores, (reps, t, s)), idx, 2).mean((1, 2))
    return float(scores.mean()), float(np.quantile(res, 0.025)), float(np.quantile(res, 0.975))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 Bootstrap：把手里的数据当成总体，重抽")
for k, n in ((40, 50), (30, 50)):
    lo, hi = bootstrap_ci(k, n)
    print(f"  {k}/{n}：bootstrap 95% 区间 [{lo:.3f}, {hi:.3f}]")

In [ ]:
print("\n第二步 Beta 后验：均匀先验 + 数据")
for k, n in ((40, 50), (30, 50)):
    d = beta_posterior(k, n)
    lo, hi = credible_interval(k, n)
    print(f"  {k}/{n} → Beta({1 + k}, {1 + n - k})：均值 {d.mean():.3f}，95% 可信区间 [{lo:.3f}, {hi:.3f}]")

In [ ]:
print("\n第三步 B 比 A 好的概率")
print(f"  P(p_40/50 > p_30/50) = {prob_b_better(40, 50, 30, 50):.3f}")
lo, hi = diff_bootstrap(40, 50, 30, 50)
print(f"  差值 bootstrap 区间 [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
print("\n第四步 多任务、多种子：分层 bootstrap")
m, lo, hi = stratified_bootstrap_mean(SCORES)
print(f"  3 个任务 × 5 个种子：平均 {m:.3f}，95% 区间 [{lo:.3f}, {hi:.3f}]")

## 3. 代码里才看得到的

1. **成功率的 bootstrap 可以直接用二项分布抽**：`diff_bootstrap` 里写成 `rng.binomial(n, k/n)`，和"有放回抽 n 个 0/1"是同一个分布，只是快得多。
2. **蒙特卡洛的误差**：40 万次抽样时 P(B > A) 的标准误约 √(0.984 × 0.016 / 400000) ≈ 0.0002，小数点后三位稳定。
3. **分层 bootstrap 用 `take_along_axis` 一次算完**：`(reps, 任务, 种子)` 三维下标，避免写 2 万次循环。

## 4. 自己改一改

1. 把先验换成 Beta(10, 10)（"我相信成功率在 0.5 附近"），40/50 的后验均值变成多少？
2. `stratified_bootstrap_mean` 里把任务 2 的分数全部加 0.3，区间的宽度变不变？为什么？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 有放回重抽几万次，分位数就是区间，什么统计量都能套；
> 均匀先验加 k 次成功，后验是贝塔 1 加 k、1 加 n 减 k；
> 两个后验各抽一个比大小，胜率就是 B 更好的概率；
> 多任务多种子分层抽，任务不动只抽种子。